# UnitedMasters ARPU Intelligence — ML Pipeline

This notebook walks through the complete ML pipeline built for the ARPU Intelligence demo. It covers:

1. **Source Data** — The raw tables feeding the pipeline
2. **Feature Engineering** — Intermediate views that shape data for models
3. **Model Training** — FORECAST and ANOMALY_DETECTION models
4. **Predictions** — Pre-computed churn scores and ARPU forecasts
5. **AI Recommendations** — LLM-generated actions per artist

---

## 1. Source Data (RAW Schema)

Three tables provide the foundation:
- **ARTISTS** — Artist profiles with tier, genre, country, monthly listeners
- **TRANSACTIONS** — Monthly royalty/streaming data by platform
- **FEATURE_ADOPTION** — Product feature usage per artist

In [ ]:
%%sql -r artists_sample
SELECT artist_id, artist_name, genre, tier, country, monthly_listeners, catalog_size
FROM ARPU_INTELLIGENCE_DEMO.RAW.ARTISTS
LIMIT 10

In [ ]:
%%sql -r transactions_sample
SELECT artist_id, period_month, platform, stream_count, royalty_usd, net_payout_usd
FROM ARPU_INTELLIGENCE_DEMO.RAW.TRANSACTIONS
ORDER BY period_month DESC
LIMIT 10

In [ ]:
%%sql -r feature_adoption_sample
SELECT artist_id, feature_name, first_used_date, sessions_last_30d, is_active_user
FROM ARPU_INTELLIGENCE_DEMO.RAW.FEATURE_ADOPTION
LIMIT 10

---
## 2. Feature Engineering (ANALYTICS Schema)

Two intermediate views transform raw data into ML-ready features:

- **ARTIST_ARPU_TIMESERIES** — Monthly ARPU per artist (feeds Forecast + Anomaly Detection)
- **ARTIST_FEATURES** — Behavioral feature matrix per artist (feeds Churn Classification)

In [ ]:
%%sql -r arpu_timeseries
-- Monthly ARPU per artist — this is the time series the forecast model trains on
SELECT *
FROM ARPU_INTELLIGENCE_DEMO.ANALYTICS.ARTIST_ARPU_TIMESERIES
ORDER BY artist_id, period_month
LIMIT 20

In [ ]:
%%sql -r artist_features
-- Behavioral feature matrix — feeds churn classification
SELECT *
FROM ARPU_INTELLIGENCE_DEMO.ANALYTICS.ARTIST_FEATURES
LIMIT 10

---
## 3. Training Data Views (ML Schema)

These views further filter the feature-engineered data into what the models actually consume:

- **FORECAST_TRAINING_DATA** — 50 artists with 10+ months history, data before July 2026
- **ANOMALY_TEST_DATA** — Same artists, data from July 2026+ (holdout)
- **CHURN_TRAINING_DATA** — Feature matrix with labels for classification

In [ ]:
%%sql -r forecast_training
-- Training data for FORECAST: 50 artists with sufficient history
-- Filters to data BEFORE July 2026 (model learns from this)
SELECT *
FROM ARPU_INTELLIGENCE_DEMO.ML.FORECAST_TRAINING_DATA
ORDER BY artist_id, period_month
LIMIT 20

In [ ]:
%%sql -r forecast_training_stats
-- How many artists and data points in training set?
SELECT
    COUNT(DISTINCT artist_id) AS num_artists,
    COUNT(*) AS total_rows,
    MIN(period_month) AS earliest_month,
    MAX(period_month) AS latest_month
FROM ARPU_INTELLIGENCE_DEMO.ML.FORECAST_TRAINING_DATA

In [ ]:
%%sql -r churn_training
-- Churn classification features + labels
SELECT *
FROM ARPU_INTELLIGENCE_DEMO.ML.CHURN_TRAINING_DATA
LIMIT 10

In [ ]:
%%sql -r churn_label_dist
-- Distribution of churn risk labels in training data
SELECT churn_risk_label, COUNT(*) AS count
FROM ARPU_INTELLIGENCE_DEMO.ML.CHURN_TRAINING_DATA
GROUP BY churn_risk_label
ORDER BY count DESC

---
## 4. Model Training

Two Snowflake ML models were trained natively — no external infrastructure needed.

### ARPU Forecast Model
```sql
CREATE SNOWFLAKE.ML.FORECAST ARPU_INTELLIGENCE_DEMO.ML.ARPU_FORECAST_MODEL(
  INPUT_DATA => SYSTEM$REFERENCE('VIEW', 'ARPU_INTELLIGENCE_DEMO.ML.FORECAST_TRAINING_DATA'),
  SERIES_COLNAME => 'ARTIST_ID',
  TIMESTAMP_COLNAME => 'PERIOD_MONTH',
  TARGET_COLNAME => 'ARPU_USD'
);
```

### Revenue Anomaly Detection Model
```sql
CREATE SNOWFLAKE.ML.ANOMALY_DETECTION ARPU_INTELLIGENCE_DEMO.ML.REVENUE_ANOMALY_MODEL(
  INPUT_DATA => SYSTEM$REFERENCE('VIEW', 'ARPU_INTELLIGENCE_DEMO.ML.FORECAST_TRAINING_DATA'),
  SERIES_COLNAME => 'ARTIST_ID',
  TIMESTAMP_COLNAME => 'PERIOD_MONTH',
  TARGET_COLNAME => 'ARPU_USD'
);
```

Both models are stored as first-class Snowflake objects with versioning.

In [ ]:
%%sql -r show_forecast_model
-- Verify the forecast model exists and check version
SHOW SNOWFLAKE.ML.FORECAST IN SCHEMA ARPU_INTELLIGENCE_DEMO.ML

In [ ]:
%%sql -r show_anomaly_model
-- Verify the anomaly detection model
SHOW SNOWFLAKE.ML.ANOMALY_DETECTION IN SCHEMA ARPU_INTELLIGENCE_DEMO.ML

---
## 5. Model Outputs

### Forecast Results
The forecast model predicts ARPU 3 months into the future per artist.

In [ ]:
%%sql -r forecast_results
-- Stored forecast predictions
SELECT *
FROM ARPU_INTELLIGENCE_DEMO.ML.FORECAST_RESULTS
ORDER BY SERIES, TS
LIMIT 20

In [ ]:
%%sql -r anomaly_results
-- Anomalies detected in revenue patterns
SELECT *
FROM ARPU_INTELLIGENCE_DEMO.ML.ANOMALY_RESULTS

---
## 6. Combined Predictions Table

The `ML.PREDICTIONS` table combines all model outputs into a single row per artist:
- Churn score and risk label (from classification)
- 90-day ARPU forecast
- Top recommended feature
- Forecast lift percentage

This is the main table consumed by both the Streamlit dashboard and the Cortex Agent.

In [ ]:
%%sql -r predictions
SELECT *
FROM ARPU_INTELLIGENCE_DEMO.ML.PREDICTIONS
ORDER BY churn_score DESC
LIMIT 15

In [ ]:
%%sql -r predictions_summary
-- Summary of predictions by churn risk level
SELECT
    churn_risk_label,
    COUNT(*) AS artist_count,
    ROUND(AVG(churn_score), 3) AS avg_churn_score,
    ROUND(AVG(arpu_forecast_next_90d), 2) AS avg_forecast_arpu,
    ROUND(AVG(forecast_lift_pct), 1) AS avg_forecast_lift_pct
FROM ARPU_INTELLIGENCE_DEMO.ML.PREDICTIONS
GROUP BY churn_risk_label
ORDER BY avg_churn_score DESC

---
## 7. AI Recommendations (Cortex AI_COMPLETE)

The final step generates personalized recommendations using `SNOWFLAKE.CORTEX.COMPLETE` (llama3.1-70b). The LLM receives each artist's metrics and returns an actionable recommendation.

This is called live in the Streamlit dashboard, but a sample is also stored:

In [ ]:
%%sql -r recommendations
SELECT * FROM ARPU_INTELLIGENCE_DEMO.ML.RECOMMENDATIONS

In [ ]:
%%sql -r live_recommendation
-- Generate a LIVE recommendation for the highest-risk artists
SELECT
    p.artist_id,
    a.artist_name,
    p.churn_risk_label,
    p.churn_score,
    SNOWFLAKE.CORTEX.COMPLETE('llama3.1-70b',
        'You are a music distribution platform analyst. Given this artist profile:\n' ||
        '- Name: ' || a.artist_name || '\n' ||
        '- Tier: ' || a.tier || '\n' ||
        '- Genre: ' || a.genre || '\n' ||
        '- Churn Score: ' || p.churn_score::VARCHAR || '\n' ||
        '- Forecast ARPU: $' || p.arpu_forecast_next_90d::VARCHAR || '\n' ||
        '- Recommended Feature: ' || p.top_recommended_feature || '\n\n' ||
        'Write a 2-sentence actionable recommendation to retain this artist and grow their revenue.'
    ) AS ai_recommendation
FROM ARPU_INTELLIGENCE_DEMO.ML.PREDICTIONS p
JOIN ARPU_INTELLIGENCE_DEMO.RAW.ARTISTS a ON p.artist_id = a.artist_id
WHERE p.churn_risk_label = 'High'
ORDER BY p.churn_score DESC
LIMIT 3

---
## Pipeline Summary

```
RAW.ARTISTS ─────────────────┐
RAW.TRANSACTIONS ────────────┼──> ANALYTICS.ARTIST_ARPU_TIMESERIES ──> ML.FORECAST_TRAINING_DATA ──> ARPU_FORECAST_MODEL
RAW.FEATURE_ADOPTION ────────┤                                         ML.ANOMALY_TEST_DATA ───────> REVENUE_ANOMALY_MODEL
                             │
                             └──> ANALYTICS.ARTIST_FEATURES ──────────> ML.CHURN_TRAINING_DATA ────> (CLASSIFICATION*)
                                                                                                          │
                                                                                                          v
                                                                                               ML.PREDICTIONS (combined)
                                                                                                          │
                                                                                                          v
                                                                                      ┌─── Streamlit Dashboard (internal)
                                                                                      └─── Cortex Agent (external)
```

*Classification model has an account-level version issue — churn scores are pre-computed in ML.PREDICTIONS as a workaround.*